# Frozen-system explainability and error analysis

This is a diagnostic phase. The 467-feature pipeline, LightGBM and XGBoost parameters, 80/20 blend, Platt mapping, and Balanced decision policy are frozen. Component TreeSHAP explains each booster in **raw-margin/log-odds space**. It does not exactly explain the probability blend, Platt transform, or business action. SHAP importance is not causal inference. External VALIDATION is used for diagnostics only; HOLDOUT is excluded.


In [1]:
import gc
import sys
import time
from pathlib import Path

import joblib
import lightgbm as lgb
import matplotlib.pyplot as plt
import mlflow
import numpy as np
import pandas as pd
import xgboost as xgb
from IPython.display import display
from sklearn.metrics import average_precision_score, roc_auc_score

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
from fraud_detection.data import merge_train_data
from fraud_detection.decisioning.costs import scenario_profiles
from fraud_detection.decisioning.policy import decide
from fraud_detection.evaluation.error_analysis import (
    amount_band_summary,
    categorical_slices,
    disagreement_summary,
    error_classes,
    join_decisions,
    representative_cases,
    risk_band_summary,
)
from fraud_detection.evaluation.explainability import aggregate_shap, top_local_drivers
from fraud_detection.features import COMPOSITE_TARGET_ENTITIES
from fraud_detection.models import prepare_engineered_matrices
from fraud_detection.models.ensemble import weighted_score_blend

started = time.perf_counter()
calibrator = joblib.load(ROOT / "models/probability_calibrator.joblib")
assert calibrator.method == "platt"
profiles = scenario_profiles()
SEED = 42
SAMPLE_SIZE = 5000
MIN_SLICE_SUPPORT = 500

Could not save font_manager cache [Errno 13] Permission denied: 'C:\\Users\\Marwan\\.matplotlib\\fontlist-v3.11.0.json.matplotlib-lock'


## Exact frozen validation reproduction

The earlier phase did not persist validation row scores. Rebuild once with the same full-TRAIN preprocessing, original numeric types, 467 predictors, LightGBM/XGBoost configurations, and 80/20 weights. The fixed identity row limit ends at VALIDATION; no HOLDOUT row is read. Stop if reference PR-AUC or ROC-AUC is not reproduced.


In [2]:
full_rows = 590540
train_end = int(full_rows * 0.70)
validation_end = train_end + int(full_rows * 0.15)
transaction_path = ROOT / "data/raw/train_transaction.csv"
sample = pd.read_csv(transaction_path, nrows=1000, low_memory=False)
dtypes = {
    name: ("float64" if pd.api.types.is_numeric_dtype(dtype) else "object")
    for name, dtype in sample.dtypes.items()
}
dtypes["TransactionID"] = "int64"
dtypes["TransactionDT"] = "int64"
dtypes["card1"] = "int64"
dtypes["isFraud"] = "int8"
del sample
transaction = pd.read_csv(transaction_path, nrows=validation_end, dtype=dtypes)
validation_max_id = int(transaction.TransactionID.iloc[-1])
identity_path = ROOT / "data/raw/train_identity.csv"
identity_rows_through_validation = 125565  # Fixed IEEE-CIS chronological split boundary.
identity = pd.read_csv(identity_path, nrows=identity_rows_through_validation, low_memory=False)
assert int(identity.TransactionID.iloc[-1]) <= validation_max_id
merged = merge_train_data(transaction, identity)
train = merged.iloc[:train_end]
validation = merged.iloc[train_end:validation_end]
del transaction, identity, merged
gc.collect()
prepared = prepare_engineered_matrices(
    train,
    validation,
    include_graph=False,
    include_composite=True,
    include_target_encoding=True,
    scaling_batch_rows=len(train),
)
positions = {name: i for i, name in enumerate(prepared.feature_names)}
baseline_names = tuple(
    name for name in prepared.feature_names if not name.endswith("_target_encoded_fraud_rate")
)
composite_te_features = tuple(
    f"{entity}_target_encoded_fraud_rate" for entity in COMPOSITE_TARGET_ENTITIES
)
selected_names = baseline_names + composite_te_features
assert len(selected_names) == 467 and len(set(selected_names)) == 467
assert "isFraud" not in selected_names and "y_true" not in selected_names
indices = np.asarray([positions[name] for name in selected_names], dtype=np.int32)
x_train, x_validation = prepared.x_train[:, indices], prepared.x_evaluation[:, indices]
y_train = train.isFraud.astype("int8")
y_validation = validation.isFraud.astype("int8")
matrix_mib = (prepared.x_train.nbytes + prepared.x_evaluation.nbytes) / 1024**2
lgbm = lgb.LGBMClassifier(
    learning_rate=0.1,
    n_estimators=400,
    num_leaves=63,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1,
    verbosity=-1,
)
lgbm.fit(x_train, y_train)
lgbm_score = lgbm.predict_proba(x_validation)[:, 1]
scale_pos_weight = float((y_train == 0).sum() / (y_train == 1).sum())
xgboost = xgb.XGBClassifier(
    learning_rate=0.1,
    max_depth=7,
    n_estimators=300,
    tree_method="hist",
    eval_metric="aucpr",
    random_state=42,
    n_jobs=-1,
    scale_pos_weight=scale_pos_weight,
)
xgboost.fit(x_train, y_train)
xgboost_score = xgboost.predict_proba(x_validation)[:, 1]
raw_validation = weighted_score_blend(
    lgbm_score, xgboost_score, lightgbm_weight=0.8, xgboost_weight=0.2
)
calibrated_validation = calibrator.transform(raw_validation)
assert abs(average_precision_score(y_validation, raw_validation) - 0.615265) < 0.000002
assert abs(roc_auc_score(y_validation, raw_validation) - 0.916224) < 0.000002

assert len(selected_names) == 467

## Component TreeSHAP on a representative sample

Use a uniform, without-replacement sample of 5,000 VALIDATION rows with seed 42. Fraud is not oversampled for global importance. Native LightGBM and XGBoost tree-contribution APIs implement TreeSHAP efficiently and return raw-margin/log-odds contributions plus a bias term. Verify additivity against each model raw margin. The final calibrated ensemble is a nonlinear scoring chain, so these are **COMPONENT MODEL EXPLANATIONS**.


In [3]:
train_edges = np.r_[
    0.0, np.unique(train.TransactionAmt.quantile([0.25, 0.50, 0.75, 0.90]).to_numpy()), np.inf
]
high_value_amount = float(train.TransactionAmt.quantile(0.90))
identity_columns = tuple(pd.read_csv(ROOT / "data/raw/train_identity.csv", nrows=0).columns)
context_columns = [
    "TransactionID",
    "isFraud",
    "TransactionAmt",
    "ProductCD",
    "DeviceType",
    "card4",
    "card6",
    "P_emaildomain",
]
context = validation.loc[:, context_columns].copy()
context["lightgbm_score"] = lgbm_score
context["xgboost_score"] = xgboost_score
context["raw_ensemble_score"] = raw_validation
decision_input = pd.DataFrame(
    {
        "TransactionID": context.TransactionID.to_numpy(),
        "calibrated_probability": calibrated_validation,
        "TransactionAmt": context.TransactionAmt.to_numpy(),
    }
)
balanced = decide(decision_input, profiles["Balanced"])
diagnostic = join_decisions(context.drop(columns="TransactionAmt"), balanced)
diagnostic["error_class"] = error_classes(diagnostic.isFraud, diagnostic.calibrated_probability)
assert diagnostic.TransactionID.is_unique
rng = np.random.default_rng(SEED)
sample_positions = np.sort(rng.choice(len(diagnostic), size=SAMPLE_SIZE, replace=False))
x_sample = np.ascontiguousarray(x_validation[sample_positions])
sample_diagnostic = diagnostic.iloc[sample_positions].reset_index(drop=True)
print("PR-AUC:", average_precision_score(diagnostic.isFraud, raw_validation))
print("ROC-AUC:", roc_auc_score(diagnostic.isFraud, raw_validation))
print("SHAP sample size:", len(sample_positions), "seed:", SEED)
print("TRAIN amount edges:", train_edges)
del prepared, x_train, train, validation, context, decision_input, balanced
# Keep only validation matrix, sampled matrix, small diagnostic context, and frozen boosters.
gc.collect()

PR-AUC: 0.6152645019628108
ROC-AUC: 0.9162239563163297
SHAP sample size: 5000 seed: 42
TRAIN amount edges: [  0.    42.95  68.95 125.   280.      inf]


36

In [4]:
lgb_contrib = lgbm.booster_.predict(x_sample, pred_contrib=True)
assert lgb_contrib.shape == (SAMPLE_SIZE, len(selected_names) + 1)
lgb_margin = lgbm.booster_.predict(x_sample, raw_score=True)
assert np.allclose(lgb_contrib.sum(axis=1), lgb_margin, atol=1e-4)
dmat = xgb.DMatrix(x_sample)
xgb_contrib = xgboost.get_booster().predict(dmat, pred_contribs=True)
assert xgb_contrib.shape == (SAMPLE_SIZE, len(selected_names) + 1)
xgb_margin = xgboost.get_booster().predict(dmat, output_margin=True)
assert np.allclose(xgb_contrib.sum(axis=1), xgb_margin, atol=1e-4)
lgb_shap, xgb_shap = lgb_contrib[:, :-1], xgb_contrib[:, :-1]
print("TreeSHAP raw-margin additivity verified for both components")

TreeSHAP raw-margin additivity verified for both components


## Global component drivers and feature families

Mean absolute SHAP is computed separately for each model in raw-margin units. Feature families are source-based and exhaustive; categorical raw columns frequency-encoded inside preprocessing retain their raw source family. Directional summaries describe model behavior in this sample, not causal fraud mechanisms. Four composite temporal target encodings reflect **matured historical fraud-risk signals**, never current-row labels.


In [5]:
lgb_features, lgb_families = aggregate_shap(lgb_shap, selected_names, identity_columns)
xgb_features, xgb_families = aggregate_shap(xgb_shap, selected_names, identity_columns)
for name, table in [("lightgbm", lgb_features), ("xgboost", xgb_features)]:
    table.to_csv(ROOT / f"reports/{name}_shap_feature_importance.csv", index=False)
family_comparison = lgb_families.merge(
    xgb_families, on="family", suffixes=("_lightgbm", "_xgboost")
)
family_comparison.to_csv(ROOT / "reports/shap_feature_family_importance.csv", index=False)
display(lgb_features.head(20), xgb_features.head(20), family_comparison)
te_names = list(composite_te_features)
te_rows = []
for model_name, values, table in [
    ("LightGBM", lgb_shap, lgb_features),
    ("XGBoost", xgb_shap, xgb_features),
]:
    for name in te_names:
        j = selected_names.index(name)
        contribution = values[:, j]
        row = table.loc[table.feature == name].iloc[0]
        te_rows.append(
            {
                "model": model_name,
                "feature": name,
                "rank": int(row["rank"]),
                "mean_abs_shap": float(row.mean_abs_shap),
                "p05_shap": float(np.quantile(contribution, 0.05)),
                "median_shap": float(np.median(contribution)),
                "p95_shap": float(np.quantile(contribution, 0.95)),
                "positive_fraction": float((contribution > 0).mean()),
            }
        )
te_table = pd.DataFrame(te_rows)
display(te_table)
te_table.to_csv(ROOT / "reports/composite_te_shap_diagnostics.csv", index=False)
common_top20 = sorted(set(lgb_features.head(20).feature) & set(xgb_features.head(20).feature))
lgb_only_top20 = sorted(set(lgb_features.head(20).feature) - set(xgb_features.head(20).feature))
xgb_only_top20 = sorted(set(xgb_features.head(20).feature) - set(lgb_features.head(20).feature))
print("Shared top-20:", common_top20)
print("LightGBM-only top-20:", lgb_only_top20)
print("XGBoost-only top-20:", xgb_only_top20)

,feature,mean_abs_shap,family,rank
0,TransactionDT,0.480729,raw_transaction,1
1,card1_addr1_p_emaildomain_target_encoded_fraud...,0.452007,composite_temporal_te,2
2,D1,0.275795,raw_transaction,3
3,C13,0.258913,raw_transaction,4
4,card1_card2_addr1_productcd_target_encoded_fra...,0.229655,composite_temporal_te,5
5,card1_addr1_deviceinfo_target_encoded_fraud_rate,0.202798,composite_temporal_te,6
6,V307,0.155604,raw_transaction,7
7,amount_vs_addr1_previous_avg,0.143394,historical_amount,8
8,C14,0.138475,raw_transaction,9
9,card1_addr1_p_emaildomain_previous_tx_count,0.131664,composite_identity,10


,feature,mean_abs_shap,family,rank
0,card1_addr1_p_emaildomain_target_encoded_fraud...,0.646491,composite_temporal_te,1
1,card1_card2_addr1_productcd_target_encoded_fra...,0.346463,composite_temporal_te,2
2,D1,0.265214,raw_transaction,3
3,C13,0.218634,raw_transaction,4
4,TransactionDT,0.210348,raw_transaction,5
5,C14,0.202552,raw_transaction,6
6,C1,0.185530,raw_transaction,7
7,amount_vs_addr1_previous_avg,0.176496,historical_amount,8
8,card1_addr1_deviceinfo_target_encoded_fraud_rate,0.167485,composite_temporal_te,9
9,card1_addr1_p_emaildomain_previous_tx_count,0.128689,composite_identity,10


,family,mean_abs_shap_lightgbm,share_lightgbm,rank_lightgbm,mean_abs_shap_xgboost,share_xgboost,rank_xgboost
0,raw_transaction,4.968117,0.624234,1,4.397761,0.602954,1
1,composite_identity,1.037855,0.130405,2,0.880643,0.120740,3
2,composite_temporal_te,0.928600,0.116677,3,1.199815,0.164500,2
3,historical_amount,0.372062,0.046749,4,0.358246,0.049117,4
4,raw_identity,0.364804,0.045837,5,0.239538,0.032842,5
5,frequency_encoding,0.287300,0.036099,6,0.217694,0.029847,6


,model,feature,rank,mean_abs_shap,p05_shap,median_shap,p95_shap,positive_fraction
0,LightGBM,card1_card2_addr1_target_encoded_fraud_rate,55,0.044139,-0.086935,0.011142,0.111609,0.6224
1,LightGBM,card1_addr1_p_emaildomain_target_encoded_fraud...,2,0.452007,-0.530516,-0.033276,1.675399,0.4600
2,LightGBM,card1_card2_addr1_productcd_target_encoded_fra...,5,0.229655,-0.338555,-0.026990,0.606042,0.4398
3,LightGBM,card1_addr1_deviceinfo_target_encoded_fraud_rate,6,0.202798,-0.179164,-0.032622,0.749877,0.3988
4,XGBoost,card1_card2_addr1_target_encoded_fraud_rate,52,0.039376,-0.113485,0.000984,0.065210,0.5140
5,XGBoost,card1_addr1_p_emaildomain_target_encoded_fraud...,1,0.646491,-1.104461,-0.536214,1.116506,0.2374
6,XGBoost,card1_card2_addr1_productcd_target_encoded_fra...,2,0.346463,-0.598130,-0.299375,0.477863,0.2326
7,XGBoost,card1_addr1_deviceinfo_target_encoded_fraud_rate,9,0.167485,-0.293319,-0.115135,0.286608,0.2566


Shared top-20: ['C1', 'C13', 'C14', 'D1', 'P_emaildomain', 'TransactionAmt', 'TransactionDT', 'V307', 'amount_vs_addr1_previous_avg', 'card1_addr1_deviceinfo_composite_frequency', 'card1_addr1_deviceinfo_target_encoded_fraud_rate', 'card1_addr1_p_emaildomain_previous_tx_count', 'card1_addr1_p_emaildomain_target_encoded_fraud_rate', 'card1_card2_addr1_productcd_target_encoded_fraud_rate', 'card1_frequency']
LightGBM-only top-20: ['C5', 'card1_addr1_p_emaildomain_composite_frequency', 'card1_card2_addr1_composite_frequency', 'card1_previous_avg_amount', 'card2_frequency']
XGBoost-only top-20: ['C11', 'D4', 'V258', 'V70', 'V91']


In [6]:
def plot_shap_summary(values, importance, filename, title):
    top = importance.head(20).iloc[::-1]
    fig, ax = plt.subplots(figsize=(9, 8))
    jitter_rng = np.random.default_rng(SEED)
    for position, feature in enumerate(top.feature):
        j = selected_names.index(feature)
        contribution = values[:, j]
        feature_values = x_sample[:, j]
        lo, hi = np.quantile(feature_values, [0.05, 0.95])
        color = (
            np.clip((feature_values - lo) / (hi - lo), 0, 1)
            if hi > lo
            else np.full(len(feature_values), 0.5)
        )
        jitter = jitter_rng.uniform(-0.30, 0.30, len(contribution))
        ax.scatter(
            contribution,
            position + jitter,
            c=color,
            cmap="viridis",
            s=2,
            alpha=0.28,
            linewidths=0,
            rasterized=True,
        )
    ax.set_yticks(range(len(top)), top.feature)
    ax.axvline(0, color="black", lw=0.8)
    ax.set(xlabel="TreeSHAP contribution to raw margin (log-odds)", title=title)
    ax.grid(axis="x", alpha=0.2)
    fig.tight_layout()
    fig.savefig(ROOT / "reports/figures" / filename, dpi=170)
    plt.close(fig)


plot_shap_summary(
    lgb_shap,
    lgb_features,
    "lightgbm_shap_summary.png",
    "LightGBM component TreeSHAP: 5,000 representative rows",
)
plot_shap_summary(
    xgb_shap,
    xgb_features,
    "xgboost_shap_summary.png",
    "XGBoost component TreeSHAP: 5,000 representative rows",
)
family_plot = family_comparison.set_index("family")[["share_lightgbm", "share_xgboost"]]
ax = family_plot.plot.barh(figsize=(9, 5), color=["#2b7bba", "#e18a3b"])
ax.set(
    xlabel="Share of total mean absolute raw-margin SHAP",
    ylabel="Feature family",
    title="Component feature-family importance",
)
ax.legend(["LightGBM", "XGBoost"])
ax.figure.tight_layout()
ax.figure.savefig(ROOT / "reports/figures/feature_family_importance.png", dpi=170)
plt.close(ax.figure)

## Diagnostic errors, amount misses, and customer friction

TP/FP/FN/TN use calibrated probability ≥ 0.50 only as a diagnostic reference cutoff, **not a production threshold**. Business friction is assessed separately through the frozen Balanced APPROVE/REVIEW/BLOCK actions. Amount bands and the high-value cutoff come only from TRAIN quantiles.


In [7]:
class_counts = diagnostic.error_class.value_counts().reindex(["TP", "FP", "FN", "TN"], fill_value=0)
class_amounts = diagnostic.groupby("error_class").TransactionAmt.agg(
    ["count", "mean", "median", "sum"]
)
amount_table = amount_band_summary(diagnostic, train_edges)
amount_table.to_csv(ROOT / "reports/error_by_amount_band.csv", index=False)
fn = diagnostic.loc[diagnostic.error_class == "FN"]
all_fraud_amount = diagnostic.loc[diagnostic.isFraud == 1, "TransactionAmt"].sum()
high_value_fn = fn.loc[fn.TransactionAmt >= high_value_amount]
fn_summary = {
    "false_negative_count": len(fn),
    "fraudulent_amount_missed": float(fn.TransactionAmt.sum()),
    "missed_amount_share": float(fn.TransactionAmt.sum() / all_fraud_amount),
    "median_missed_amount": float(fn.TransactionAmt.median()),
    "mean_missed_amount": float(fn.TransactionAmt.mean()),
    "p90_missed_amount": float(fn.TransactionAmt.quantile(0.90)),
    "p95_missed_amount": float(fn.TransactionAmt.quantile(0.95)),
    "max_missed_amount": float(fn.TransactionAmt.max()),
    "median_fn_probability": float(fn.calibrated_probability.median()),
    "p90_fn_probability": float(fn.calibrated_probability.quantile(0.90)),
    "high_value_threshold": high_value_amount,
    "high_value_fn_count": len(high_value_fn),
    "high_value_fn_amount": float(high_value_fn.TransactionAmt.sum()),
    "high_value_fn_approve_count": int((high_value_fn.decision == "APPROVE").sum()),
    "fraud_approved_count": int(
        ((diagnostic.isFraud == 1) & (diagnostic.decision == "APPROVE")).sum()
    ),
    "fraud_approved_amount": float(
        diagnostic.loc[
            (diagnostic.isFraud == 1) & (diagnostic.decision == "APPROVE"), "TransactionAmt"
        ].sum()
    ),
}
fn_decisions = fn.decision.value_counts().to_dict()
legitimate = diagnostic.loc[diagnostic.isFraud == 0]
legitimate_intervened = legitimate.loc[legitimate.decision.isin(["REVIEW", "BLOCK"])]
friction_summary = {
    "legitimate_review_count": int((legitimate.decision == "REVIEW").sum()),
    "legitimate_review_rate": float((legitimate.decision == "REVIEW").mean()),
    "legitimate_block_count": int((legitimate.decision == "BLOCK").sum()),
    "legitimate_block_rate": float((legitimate.decision == "BLOCK").mean()),
    "legitimate_high_risk_count": int((legitimate.calibrated_probability >= 0.50).sum()),
    "intervened_amount_median": float(legitimate_intervened.TransactionAmt.median()),
    "intervened_amount_p90": float(legitimate_intervened.TransactionAmt.quantile(0.90)),
    "intervened_probability_median": float(legitimate_intervened.calibrated_probability.median()),
    "intervened_probability_p90": float(
        legitimate_intervened.calibrated_probability.quantile(0.90)
    ),
}
display(class_counts, class_amounts, amount_table)
print("False negatives:", fn_summary, "Balanced decisions:", fn_decisions)
print("Legitimate friction:", friction_summary)

error_class
TP     1167
FP      179
FN     1875
TN    85360
Name: count, dtype: int64

,count,mean,median,sum
error_class,,,,
FN,1875,185.308473,88.500,3.474534e+05
FP,179,124.607642,59.144,2.230477e+04
TN,85360,133.883333,67.950,1.142828e+07
TP,1167,128.067011,64.554,1.494542e+05


,band,amount_min,amount_max,count,fraud_prevalence,TP,FP,FN,TN,recall,precision,false_positive_rate,false_negative_rate,fraud_amount,missed_fraud_amount
0,0,0.00,42.95,21712,0.037076,357,63,448,20844,0.443478,0.850000,0.003013,0.556522,19913.589,10945.847
1,1,42.95,68.95,23107,0.025706,245,30,349,22483,0.412458,0.890909,0.001333,0.587542,32030.954,19092.109
2,2,68.95,125.00,22127,0.031455,312,39,384,21392,0.448276,0.888889,0.001820,0.551724,65896.507,37251.532
3,3,125.00,280.00,12976,0.038147,145,32,350,12449,0.292929,0.819209,0.002564,0.707071,91153.161,66248.787
4,4,280.00,inf,8659,0.052200,108,15,344,8192,0.238938,0.878049,0.001828,0.761062,287913.377,213915.111


False negatives: {'false_negative_count': 1875, 'fraudulent_amount_missed': 347453.386, 'missed_amount_share': 0.6992313951140549, 'median_missed_amount': 88.5, 'mean_missed_amount': 185.30847253333334, 'p90_missed_amount': 445.0, 'p95_missed_amount': 678.3849999999978, 'max_missed_amount': 3822.95, 'median_fn_probability': 0.10147685687336051, 'p90_fn_probability': 0.3754382583472086, 'high_value_threshold': 280.0, 'high_value_fn_count': 344, 'high_value_fn_amount': 213915.111, 'high_value_fn_approve_count': 21, 'fraud_approved_count': 709, 'fraud_approved_amount': 49739.884} Balanced decisions: {'REVIEW': 1004, 'APPROVE': 707, 'BLOCK': 164}
Legitimate friction: {'legitimate_review_count': 11865, 'legitimate_review_rate': 0.13870865920808054, 'legitimate_block_count': 475, 'legitimate_block_rate': 0.00555302259787933, 'legitimate_high_risk_count': 179, 'intervened_amount_median': 226.0, 'intervened_amount_p90': 884.0, 'intervened_probability_median': 0.05388258993015413, 'intervened_p

In [8]:
fig, ax = plt.subplots(figsize=(8, 5))
x = np.arange(len(amount_table))
ax.bar(x - 0.18, amount_table.TP, width=0.36, label="TP", color="#3176ab")
ax.bar(x + 0.18, amount_table.FN, width=0.36, label="FN", color="#c65d5d")
ax.set_xticks(
    x,
    [
        f"{v:.0f}–{w:.0f}" if np.isfinite(w) else f"{v:.0f}+"
        for v, w in zip(amount_table.amount_min, amount_table.amount_max, strict=True)
    ],
)
ax.set(
    xlabel="TRAIN-derived amount band",
    ylabel="Fraud transaction count",
    title="Diagnostic fraud outcomes by transaction amount",
)
ax.legend()
fig.tight_layout()
fig.savefig(ROOT / "reports/figures/error_by_amount_band.png", dpi=170)
plt.close(fig)
fig, ax = plt.subplots(figsize=(8, 5))
fraud_amounts = diagnostic.loc[diagnostic.isFraud == 1, "TransactionAmt"]
limits = np.geomspace(max(1, fraud_amounts.min()), max(2, fraud_amounts.max()), 35)
ax.hist(fraud_amounts, bins=limits, alpha=0.45, label="All validation fraud", color="#3176ab")
ax.hist(fn.TransactionAmt, bins=limits, alpha=0.65, label="False negatives", color="#c65d5d")
ax.axvline(high_value_amount, color="black", ls="--", label="TRAIN p90 amount")
ax.set_xscale("log")
ax.set(
    xlabel="TransactionAmt (log scale)",
    ylabel="Fraud count",
    title="False-negative amount distribution",
)
ax.legend()
fig.tight_layout()
fig.savefig(ROOT / "reports/figures/false_negative_amount_analysis.png", dpi=170)
plt.close(fig)

## Calibrated risk bands and supported categorical slices

Fixed bands are 0–1%, 1–2%, 2–5%, 5–10%, 10–20%, 20–40%, and 40%+. A slice is reported only with at least 500 validation transactions. These are descriptive checks, not policy changes or causal conclusions.


In [9]:
risk_table = risk_band_summary(diagnostic)
risk_table.to_csv(ROOT / "reports/calibrated_risk_band_diagnostics.csv", index=False)
slice_tables = [
    categorical_slices(diagnostic, column, min_support=MIN_SLICE_SUPPORT)
    for column in ("ProductCD", "DeviceType", "card4", "card6", "P_emaildomain")
]
slice_table = pd.concat([table for table in slice_tables if not table.empty], ignore_index=True)
slice_table.to_csv(ROOT / "reports/supported_categorical_slices.csv", index=False)
display(risk_table, slice_table)
sample_intervention = (
    sample_diagnostic.isFraud.eq(0) & sample_diagnostic.decision.isin(["REVIEW", "BLOCK"])
).to_numpy()
friction_lgb_features, _ = aggregate_shap(
    lgb_shap[sample_intervention], selected_names, identity_columns
)
friction_xgb_features, _ = aggregate_shap(
    xgb_shap[sample_intervention], selected_names, identity_columns
)
print("Representative-sample legitimate intervention rows:", sample_intervention.sum())
display(friction_lgb_features.head(10), friction_xgb_features.head(10))
fig, ax = plt.subplots(figsize=(9, 5))
x = np.arange(len(risk_table))
ax.bar(x, risk_table.observed_fraud_rate, color="#3176ab", width=0.42, label="Observed fraud rate")
ax.plot(
    x,
    risk_table.mean_calibrated_probability,
    marker="o",
    color="#d57432",
    label="Mean calibrated probability",
)
ax.set_xticks(x, risk_table.risk_band)
ax.set(
    xlabel="Calibrated risk band",
    ylabel="Probability / observed fraud rate",
    title="Risk band reliability on external VALIDATION",
)
ax.legend()
fig.tight_layout()
fig.savefig(ROOT / "reports/figures/risk_band_observed_fraud.png", dpi=170)
plt.close(fig)

,risk_band,count,fraud_count,observed_fraud_rate,mean_calibrated_probability,transaction_amount,fraud_amount,approve_rate,review_rate,block_rate
0,0–1%,44675,159,0.003559,0.003851,4859626.770,22135.590,0.993688,0.006312,0.000000
1,1–2%,15306,151,0.009865,0.014462,2073109.097,26670.532,0.928721,0.071279,0.000000
2,2–5%,16772,331,0.019735,0.031350,2866084.957,62591.314,0.735810,0.263117,0.001073
3,5–10%,6231,291,0.046702,0.068798,1191822.701,61867.714,0.373616,0.615792,0.010592
4,10–20%,2773,386,0.139199,0.137276,540845.356,83120.490,0.182834,0.765597,0.051569
5,20–40%,1227,410,0.334148,0.276456,204093.471,66120.700,0.088020,0.747351,0.164629
6,40%+,1597,1314,0.822793,0.738680,211911.342,174401.248,0.010019,0.231684,0.758297


,column,category,count,fraud_prevalence,mean_calibrated_risk,recall_at_0_50,legitimate_intervention_rate,fraud_amount_intervention_rate
0,ProductCD,C,9125,0.126027,0.118317,0.591304,0.105831,0.902537
1,ProductCD,H,2350,0.059574,0.070473,0.457143,0.173303,0.936457
2,ProductCD,R,3365,0.047548,0.059021,0.562500,0.274883,0.982720
3,ProductCD,S,1450,0.073793,0.053490,0.420561,0.046165,0.845712
4,ProductCD,W,72291,0.020542,0.023876,0.193939,0.143632,0.888201
5,DeviceType,<MISSING>,73350,0.021172,0.024610,0.196394,0.143892,0.887983
6,DeviceType,desktop,8741,0.097128,0.098077,0.633687,0.159022,0.936142
7,DeviceType,mobile,6490,0.098613,0.090139,0.506250,0.128889,0.933175
8,card4,american express,751,0.038615,0.059178,0.482759,0.329640,0.965565
9,card4,discover,963,0.126687,0.076234,0.237705,0.399524,0.977094


Representative-sample legitimate intervention rows: 657


,feature,mean_abs_shap,family,rank
0,card1_addr1_p_emaildomain_target_encoded_fraud...,0.630326,composite_temporal_te,1
1,TransactionDT,0.481128,raw_transaction,2
2,C13,0.274526,raw_transaction,3
3,card1_addr1_deviceinfo_target_encoded_fraud_rate,0.271930,composite_temporal_te,4
4,card1_card2_addr1_productcd_target_encoded_fra...,0.268181,composite_temporal_te,5
5,C14,0.265159,raw_transaction,6
6,D1,0.242794,raw_transaction,7
7,amount_vs_addr1_previous_avg,0.224128,historical_amount,8
8,P_emaildomain,0.170512,raw_transaction,9
9,card1_addr1_p_emaildomain_previous_tx_count,0.145340,composite_identity,10


,feature,mean_abs_shap,family,rank
0,card1_addr1_p_emaildomain_target_encoded_fraud...,0.555143,composite_temporal_te,1
1,card1_card2_addr1_productcd_target_encoded_fra...,0.303535,composite_temporal_te,2
2,C14,0.282655,raw_transaction,3
3,amount_vs_addr1_previous_avg,0.198333,historical_amount,4
4,C1,0.181741,raw_transaction,5
5,TransactionDT,0.180363,raw_transaction,6
6,D1,0.163317,raw_transaction,7
7,C13,0.157085,raw_transaction,8
8,card1_addr1_deviceinfo_target_encoded_fraud_rate,0.143296,composite_temporal_te,9
9,card1_addr1_p_emaildomain_previous_tx_count,0.137145,composite_identity,10


## LightGBM versus XGBoost disagreement

Examine absolute component-score differences and percentile-rank differences. The highest-disagreement group is the top 10% by absolute score gap, defined on VALIDATION for diagnostics only. Compare fraud prevalence, diagnostic false-negative frequency, and fraud amount. This investigates component complementarity without changing ensemble weights.


In [10]:
disagreement = disagreement_summary(
    diagnostic.isFraud,
    diagnostic.TransactionAmt,
    diagnostic.lightgbm_score,
    diagnostic.xgboost_score,
    diagnostic.calibrated_probability,
)
gap = np.abs(diagnostic.lightgbm_score.to_numpy() - diagnostic.xgboost_score.to_numpy())
high_gap = gap >= disagreement["high_gap_threshold"]
print(disagreement)
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
axes[0].hist(gap, bins=45, color="#657aa6", alpha=0.85)
axes[0].axvline(
    disagreement["high_gap_threshold"], color="#c65d5d", ls="--", label="Top-decile cutoff"
)
axes[0].set(
    xlabel="Absolute component probability gap", ylabel="Transactions", title="Score disagreement"
)
axes[0].legend()
axes[1].scatter(
    diagnostic.lightgbm_score.iloc[::35],
    diagnostic.xgboost_score.iloc[::35],
    s=5,
    alpha=0.2,
    color="#3176ab",
)
axes[1].plot([0, 1], [0, 1], ls="--", color="gray")
axes[1].set(
    xlabel="LightGBM probability", ylabel="XGBoost probability", title="Component score comparison"
)
fig.tight_layout()
fig.savefig(ROOT / "reports/figures/model_disagreement_analysis.png", dpi=170)
plt.close(fig)

{'pearson': 0.8967599430353742, 'spearman': 0.9061092065511711, 'mean_absolute_gap': 0.050877491442805345, 'mean_absolute_rank_gap': 0.09132422756820806, 'p90_absolute_rank_gap': 0.20791140312256573, 'rank_gap_ge_0_20_rate': 0.11064449486910286, 'median_absolute_gap': 0.020174876197656105, 'p90_absolute_gap': 0.14170550866866305, 'high_gap_threshold': 0.14170550866866305, 'high_gap_count': 8859, 'high_gap_fraud_rate': 0.08432102946156451, 'high_gap_false_negative_rate': 0.08149904052376114, 'high_gap_fraud_amount': 148272.75900000002, 'xgboost_higher_on_lgb_low_fraud_rate': 0.8349885408708938}


## Anonymized local cases: model versus business explanation

Cases are selected deterministically from error class, calibrated risk, action, and TRAIN-defined high-value cutoff. IDs below are report-local aliases. SHAP drivers explain the LightGBM and XGBoost **component raw margins** separately. Expected action costs explain the frozen Balanced business choice; they are not SHAP values.


In [11]:
case_positions = representative_cases(diagnostic, high_value_amount=high_value_amount)
unique_positions = list(dict.fromkeys(case_positions.values()))
local_matrix = np.ascontiguousarray(x_validation[unique_positions])
local_lgb = lgbm.booster_.predict(local_matrix, pred_contrib=True)[:, :-1]
local_xgb = xgboost.get_booster().predict(xgb.DMatrix(local_matrix), pred_contribs=True)[:, :-1]
local_lookup = {position: i for i, position in enumerate(unique_positions)}
case_rows = []
for number, (kind, position) in enumerate(case_positions.items(), 1):
    row = diagnostic.iloc[position]
    j = local_lookup[position]
    case_rows.append(
        {
            "case": f"C{number:02d}",
            "kind": kind,
            "true_fraud": int(row.isFraud),
            "lightgbm_score": float(row.lightgbm_score),
            "xgboost_score": float(row.xgboost_score),
            "raw_ensemble_score": float(row.raw_ensemble_score),
            "calibrated_probability": float(row.calibrated_probability),
            "TransactionAmt": float(row.TransactionAmt),
            "ProductCD": str(row.ProductCD),
            "DeviceType": str(row.DeviceType),
            "card4": str(row.card4),
            "card6": str(row.card6),
            "expected_approve_cost": float(row.expected_cost_approve),
            "expected_review_cost": float(row.expected_cost_review),
            "expected_block_cost": float(row.expected_cost_block),
            "decision": row.decision,
            "lightgbm_drivers": ", ".join(
                f"{name} ({value:+.3f})"
                for name, value in top_local_drivers(local_lgb[j], selected_names)
            ),
            "xgboost_drivers": ", ".join(
                f"{name} ({value:+.3f})"
                for name, value in top_local_drivers(local_xgb[j], selected_names)
            ),
        }
    )
case_table = pd.DataFrame(case_rows)
assert not case_table.columns.isin(["TransactionID"]).any()
display(case_table)

,case,kind,true_fraud,lightgbm_score,xgboost_score,raw_ensemble_score,calibrated_probability,TransactionAmt,ProductCD,DeviceType,card4,card6,expected_approve_cost,expected_review_cost,expected_block_cost,decision,lightgbm_drivers,xgboost_drivers
0,C01,high_risk_tp,1,9.999247e-01,0.999872,9.999143e-01,0.997197,25.268,C,mobile,visa,debit,25.197177,8.042238,0.570072,BLOCK,card1_addr1_p_emaildomain_target_encoded_fraud...,"V258 (+1.370), C1 (+1.095), card1_addr1_p_emai..."
1,C02,false_positive,0,9.992852e-01,0.998310,9.990902e-01,0.982409,176.448,C,desktop,mastercard,credit,173.344071,37.686405,0.939779,BLOCK,"V258 (+1.807), C1 (+1.454), card1_addr1_p_emai...","V258 (+1.559), C1 (+1.132), card1_addr1_p_emai..."
2,C03,false_negative,1,3.763955e-04,0.000998,5.006745e-04,0.000599,35.950,W,nan,mastercard,debit,0.021545,4.003710,25.485018,APPROVE,"C5 (-0.577), TransactionDT (-0.473), card1_add...",card1_addr1_p_emaildomain_target_encoded_fraud...
3,C04,low_risk_tn,0,3.032579e-07,0.000003,8.716955e-07,0.000005,30.924,C,desktop,mastercard,credit,0.000142,4.000024,25.499885,APPROVE,"id_17 (-1.336), V55 (-1.153), D13 (-1.098), V2...","card3 (-1.087), D13 (-0.811), id_17 (-0.686), ..."
4,C05,review,0,9.794195e-01,0.991835,9.819027e-01,0.840995,6.394,C,desktop,visa,debit,5.377324,4.234470,4.475117,REVIEW,card1_addr1_p_emaildomain_target_encoded_fraud...,"C1 (+1.119), card1_addr1_p_emaildomain_target_..."
5,C06,block,1,9.999247e-01,0.999872,9.999143e-01,0.997197,25.268,C,mobile,visa,debit,25.197177,8.042238,0.570072,BLOCK,card1_addr1_p_emaildomain_target_encoded_fraud...,"V258 (+1.370), C1 (+1.095), card1_addr1_p_emai..."
6,C07,high_value_fraud_miss,1,2.508598e-01,0.250224,2.507327e-01,0.089321,3822.950,W,nan,visa,credit,341.468422,72.204364,23.266983,BLOCK,"amount_vs_addr1_previous_avg (+0.440), card1_c...","amount_vs_addr1_previous_avg (+0.309), card1_a..."


In [12]:
direction_rows = []
for model_name, values, table in [
    ("LightGBM", lgb_shap, lgb_features),
    ("XGBoost", xgb_shap, xgb_features),
]:
    for feature in table.head(15).feature:
        j = selected_names.index(feature)
        correlation = pd.Series(x_sample[:, j]).corr(pd.Series(values[:, j]), method="spearman")
        direction_rows.append(
            {"model": model_name, "feature": feature, "feature_value_shap_spearman": correlation}
        )
direction_table = pd.DataFrame(direction_rows)
tracking_uri = f"sqlite:///{(ROOT / 'model_comparison.db').resolve().as_posix()}"
mlflow.set_tracking_uri(tracking_uri)
mlflow.set_experiment("fraud-detection-explainability-error-analysis")
with mlflow.start_run(run_name="frozen-validation-diagnostics") as run:
    mlflow.set_tags(
        {
            "stage": "explainability-error-analysis",
            "dataset": "ieee-cis",
            "model": "frozen-calibrated-ensemble",
        }
    )
    mlflow.log_params(
        {
            "shap_sample_size": SAMPLE_SIZE,
            "sample_seed": SEED,
            "shap_output_space": "raw_margin_log_odds",
            "minimum_slice_support": MIN_SLICE_SUPPORT,
            "lightgbm_top_feature": lgb_features.iloc[0].feature,
            "xgboost_top_feature": xgb_features.iloc[0].feature,
        }
    )
    mlflow.log_metrics(
        {
            "validation_pr_auc": average_precision_score(diagnostic.isFraud, raw_validation),
            "validation_roc_auc": roc_auc_score(diagnostic.isFraud, raw_validation),
            "false_negative_count": fn_summary["false_negative_count"],
            "fraudulent_amount_missed": fn_summary["fraudulent_amount_missed"],
            "false_positive_count": int(class_counts["FP"]),
            "legitimate_review_rate": friction_summary["legitimate_review_rate"],
            "legitimate_block_rate": friction_summary["legitimate_block_rate"],
            "high_disagreement_fraud_rate": disagreement["high_gap_fraud_rate"],
            "model_pearson": disagreement["pearson"],
            "model_spearman": disagreement["spearman"],
        }
    )
    for model, families in [("lightgbm", lgb_families), ("xgboost", xgb_families)]:
        mlflow.log_metrics(
            {
                f"{model}_family_{row.family}_share": float(row.share)
                for row in families.itertuples()
            }
        )
    for filename in (
        "lightgbm_shap_summary.png",
        "xgboost_shap_summary.png",
        "feature_family_importance.png",
        "error_by_amount_band.png",
        "risk_band_observed_fraud.png",
        "false_negative_amount_analysis.png",
        "model_disagreement_analysis.png",
    ):
        mlflow.log_artifact(str(ROOT / "reports/figures" / filename), artifact_path="figures")
    run_id = run.info.run_id
print("MLflow run:", run_id)

2026/10/11 11:30:23 INFO mlflow.tracking.fluent: Experiment with name 'fraud-detection-explainability-error-analysis' does not exist. Creating a new experiment.


MLflow run: 94961ff053e04c7a8fbe782567b96430


In [13]:
def markdown_table(frame, columns):
    lines = ["| " + " | ".join(columns) + " |", "| " + " | ".join(["---"] * len(columns)) + " |"]
    for row in frame.loc[:, columns].itertuples(index=False, name=None):
        lines.append(
            "| " + " | ".join(f"{v:.5f}" if isinstance(v, float) else str(v) for v in row) + " |"
        )
    return "\n".join(lines)


class_table = class_amounts.reset_index().rename(columns={"error_class": "class"})
class_table["diagnostic_class"] = class_table.iloc[:, 0]
class_table = class_table.loc[:, ["diagnostic_class", "count", "mean", "median", "sum"]]
summary_lines = "\n".join(f"- {key}: {value}" for key, value in fn_summary.items())
friction_lines = "\n".join(f"- {key}: {value}" for key, value in friction_summary.items())
disagreement_lines = "\n".join(f"- {key}: {value}" for key, value in disagreement.items())
case_columns = [
    "case",
    "kind",
    "true_fraud",
    "lightgbm_score",
    "xgboost_score",
    "raw_ensemble_score",
    "calibrated_probability",
    "TransactionAmt",
    "ProductCD",
    "DeviceType",
    "card4",
    "card6",
    "lightgbm_drivers",
    "xgboost_drivers",
    "expected_approve_cost",
    "expected_review_cost",
    "expected_block_cost",
    "decision",
]
report = f"""# Explainability and error analysis of the frozen system

## Frozen system and validation reproduction

The analyzed system uses 467 existing predictors, frozen LightGBM and XGBoost parameters, an 80/20 probability blend, the TRAIN-OOF-fitted Platt calibrator, and the previously frozen Balanced illustrative decision profile. Rebuilt external VALIDATION PR-AUC = {average_precision_score(diagnostic.isFraud, raw_validation):.6f}; ROC-AUC = {roc_auc_score(diagnostic.isFraud, raw_validation):.6f}. These reproduce the champion reference. Diagnostics did not change any model, feature, calibrator, ensemble weight, scenario, or policy. HOLDOUT was not loaded, predicted, explained, or evaluated.

## Component TreeSHAP scope and sample

Native LightGBM and XGBoost tree-contribution APIs calculate TreeSHAP separately in **raw-margin/log-odds space**, with additivity checked against each model's raw margin. The global sample is {SAMPLE_SIZE:,} uniformly sampled VALIDATION rows without replacement using NumPy seed {SEED}; fraud was not oversampled. The final score is component probabilities → 80/20 blend → Platt calibration → expected-cost action. These SHAP values are **COMPONENT MODEL EXPLANATIONS**, not an exact explanation of the calibrated ensemble or business decision. SHAP importance is not causal inference.

## Top LightGBM features

{markdown_table(lgb_features.head(15), ["rank", "feature", "family", "mean_abs_shap"])}

## Top XGBoost features

{markdown_table(xgb_features.head(15), ["rank", "feature", "family", "mean_abs_shap"])}

Features in both top 20: {common_top20}. LightGBM-only top 20: {lgb_only_top20}. XGBoost-only top 20: {xgb_only_top20}. Signed SHAP direction versus standardized feature value was summarized with sample Spearman correlations, not causal effects:

{markdown_table(direction_table, ["model", "feature", "feature_value_shap_spearman"])}

## Feature-family and temporal target-encoding importance

{markdown_table(family_comparison, ["family", "mean_abs_shap_lightgbm", "share_lightgbm", "rank_lightgbm", "mean_abs_shap_xgboost", "share_xgboost", "rank_xgboost"])}

The four retained composite temporal target-encoding features are **matured historical fraud-risk signals**; no current-row fraud label enters their validation values. Individual raw-margin contributions and global ranks:

{markdown_table(te_table, ["model", "feature", "rank", "mean_abs_shap", "p05_shap", "median_shap", "p95_shap", "positive_fraction"])}

## Diagnostic 0.50 error classes and transaction amount

The 0.50 **calibrated** probability cutoff is for TP/FP/FN/TN diagnostics only, not a production threshold. The actual business policy is analyzed separately.

{markdown_table(class_table, ["diagnostic_class", "count", "mean", "median", "sum"])}

Amount-band boundaries come from TRAIN quantiles {train_edges.tolist()}. Counts, error rates, and missed fraud amount:

{markdown_table(amount_table, ["band", "amount_min", "amount_max", "count", "fraud_prevalence", "TP", "FP", "FN", "TN", "recall", "precision", "false_positive_rate", "false_negative_rate", "fraud_amount", "missed_fraud_amount"])}

## High-value fraud misses and customer friction

False-negative details:

{summary_lines}

Balanced decisions among diagnostic false negatives: {fn_decisions}. A model false negative can still enter REVIEW or BLOCK because decisioning uses amount-dependent expected costs. High-value uses the TRAIN 90th-percentile amount cutoff. Legitimate customer friction:

{friction_lines}

Features most prominent within the representative sample's legitimate REVIEW/BLOCK rows (descriptive, noncausal): LightGBM {friction_lgb_features.head(5).feature.tolist()}; XGBoost {friction_xgb_features.head(5).feature.tolist()}.

## Calibrated risk bands and supported slices

{markdown_table(risk_table, ["risk_band", "count", "fraud_count", "observed_fraud_rate", "mean_calibrated_probability", "transaction_amount", "fraud_amount", "approve_rate", "review_rate", "block_rate"])}

Only category groups with at least {MIN_SLICE_SUPPORT} validation transactions are shown. Selected supported groups (highest count first):

{markdown_table(slice_table.sort_values("count", ascending=False).head(20), ["column", "category", "count", "fraud_prevalence", "mean_calibrated_risk", "recall_at_0_50", "legitimate_intervention_rate", "fraud_amount_intervention_rate"])}

The complete supported-slice table is reports/supported_categorical_slices.csv. Slice differences did not change the policy.

## Component disagreement

{disagreement_lines}

High disagreement means the top 10% of absolute LightGBM–XGBoost score gaps on VALIDATION. The XGBoost-higher rate among fraud rows scored below 0.50 by LightGBM checks whether XGBoost adds signal there; it does not establish a causal or independently deployable benefit. Earlier OOF ensemble evidence selected the frozen 80/20 blend; no weight is reconsidered here.

## Anonymized local cases

Case IDs are report-local aliases; no original TransactionID is displayed. The model columns list separate signed raw-margin component SHAP drivers. The final four cost/action columns explain the Balanced business decision.

{markdown_table(case_table, case_columns)}

## Limitations and future monitoring implications

TreeSHAP samples model behavior, not fraud causality. Component raw-margin SHAP cannot be added across boosters to explain the calibrated blend. A uniform 5,000-row sample supports global diagnostics but rare segments and local examples have higher variance. REVIEW capture uses illustrative effectiveness; actual reviewer outcomes are unavailable. The 0.50 error cutoff is diagnostic only. Future monitoring should watch calibration by risk and amount band, high-value misses, legitimate interventions, category support, and component disagreement. No monitoring system, model change, or new business rule was implemented.

## Runtime, memory, MLflow

Notebook runtime: {time.perf_counter() - started:.1f} seconds. Prepared feature-matrix estimate: {matrix_mib:.1f} MiB; SHAP matrices contain {SAMPLE_SIZE:,} × {len(selected_names)} values per model. MLflow experiment: fraud-detection-explainability-error-analysis; tracking URI: {tracking_uri}; run ID: {run_id}. Seven figures are under reports/figures/ and logged as run artifacts.
"""
(ROOT / "reports/explainability_error_analysis_summary.md").write_text(report, encoding="utf-8")
print("Report written; runtime seconds:", time.perf_counter() - started)

Report written; runtime seconds: 220.36450579999655
